> Projeto Desenvolve <br>
Programação Intermediária com Python <br>
Profa. Camila Laranjeira (mila@projetodesenvolve.com.br) <br>

# 3.11 - Data Model

## Exercícios

#### Q1. `dataclass`
Exercício adaptado de [codechalleng.es/bites/154/](https://codechalleng.es/bites/154/) e [codechalleng.es/bites/320/](https://codechalleng.es/bites/320/).

Neste desafio, você deve escrever uma `dataclass` chamada `Bite` que gerencia 3 atributos: `number`, `title` e `level`. Seus tipos são:
* `number` - `int`, 
* `title` - `str`, 
* `level` -  classe `Enum` chamada `BiteLevel` com os atributos `Beginner`, `Intermediate`, `Advanced`. 

Exemplo de dado: `{'number': 154, 'title': 'Escreva uma dataclass', 'level': BiteLevel.Intermediate}`

As características dessa classe são:
* O atributo`level` tem um valor padrão `BiteLevel.Beginner`
* Uma coleção de objetos `Bite` tem que ser ordenável somente pelo atributo `number`
* Implemente o método especial `__str__` para imprimir o Bite na forma `f'{number} - {title} ({level})'`

Teste sua classe executando o seguinte código:
```python
bites = []
bites.append(Bite(154, 'Escreva uma dataclass', 'Intermediate'))
bites.append(Bite(1, 'Some n valores'))
bites.append(Bite(37, 'Reescreva um loop com recursão', 'Intermediate'))

for b in bites.sort(): print(b)
# Ordem esperada na saída:
# 1 - Some n valores (Beginner)
# 37 - Reescreva um loop com recursão (Intermediate)
# 154 - Escreva uma dataclass (Intermediate)
```

In [ ]:
#### Escreva sua resposta aqui
from dataclasses import dataclass, field
from enum import Enum

# 1. Definição da classe Enum para os níveis
class BiteLevel(Enum):
    Beginner = 'Beginner'
    Intermediate = 'Intermediate'
    Advanced = 'Advanced'
    
    # Método para facilitar a impressão limpa do valor do Enum
    def __str__(self):
        return self.value

# 2. Definição da dataclass ordenada apenas pelo atributo 'number'
@dataclass(order=True)
class Bite:
    # Este atributo define a ordenação principal
    number: int
    
    # Retiramos 'title' e 'level' da comparação/ordenação usando field(compare=False)
    title: str = field(compare=False)
    level: BiteLevel = field(default=BiteLevel.Beginner, compare=False)

    def __post_init__(self):
        # Conversor automático: se o usuário passar uma string (ex: 'Intermediate'), 
        # nós convertemos para o membro correspondente do Enum (BiteLevel.Intermediate)
        if isinstance(self.level, str):
            self.level = BiteLevel[self.level]

    # 3. Implementação do formato de impressão customizado
    def __str__(self):
        return f"{self.number} - {self.title} ({self.level})"
    
bites = []
bites.append(Bite(154, 'Escreva uma dataclass', 'Intermediate'))
bites.append(Bite(1, 'Some n valores'))
bites.append(Bite(37, 'Loop com recursão', 'Intermediate'))

# Ordena a lista original (o método .sort() do Python ordena in-place e retorna None)
bites.sort()

# Imprime os resultados ordenados
for b in bites:
    print(b)

#### Q2. `Pydantic`
> Adaptada desse [tutorial de Pydantic](https://github.com/adonath/scipy-2023-pydantic-tutorial/tree/main) criado por [Axel Donath](https://github.com/adonath) e [Nick Langellier](https://github.com/nlangellier).

Observe a seguinte lista de observações da previsão do tempo em Murmansk, Russia.
```python
data_samples = [
    {
        "date": "2023-05-20",
        "temperature": 62.2,
        "isCelsius": False,
        "airQualityIndex": "24",
        "sunriseTime": "01:26",
        "sunsetTime": "00:00",
    },
    {
        "date": "2023-05-21",
        "temperature": "64.4",
        "isCelsius": "not true",
        "airQualityIndex": 23,
        "sunriseTime": "01:10",
        "sunsetTime": "00:16",
    },
    {
        "date": "2023-05-22",
        "temperature": 14.4,
        "airQualityIndex": 21,
    },
]
```

Escreva um script que calcule e imprima a temperatura média (em Celsius) em Murmansk para as datas fornecidas. Em seu script, você deve incluir um modelo Pydantic que registre com sucesso todos os elementos dados. Note que:

* Algumas amostras estão faltando dados. Você deve decidir quando o atributo pode ter um valor padrão ou quando definí-lo como opcional (`typing.Optional`). 
* Você precisará implementar pelo menos um validador de campo para transformar atributos. Dica: teste primeiro quais vão falhar :)



In [ ]:
#### Escreva sua resposta aqui
from datetime import date, time
from typing import Optional
from pydantic import BaseModel, Field, field_validator

# 1. Definição do modelo Pydantic
class WeatherSample(BaseModel):
    date: date
    temperature: float
    # Definido como opcional com padrão True (caso falte, assumimos Celsius)
    isCelsius: Optional[bool] = True  
    airQualityIndex: int
    # Campos ausentes no terceiro registro definidos como opcionais
    sunriseTime: Optional[time] = None
    sunsetTime: Optional[time] = None

    # Validador de campo customizado para limpar e converter os dados inconsistentes
    @field_validator('isCelsius', mode='before')
    @classmethod
    def ajustar_booleano(cls, v):
        if isinstance(v, str):
            # Trata strings "not true", "false", etc.
            if v.strip().lower() in ('not true', 'false', '0'):
                return False
            if v.strip().lower() in ('true', '1'):
                return True
        return v

    # Método interno para garantir que a saída seja sempre obtida em Celsius
    @property
    def temperatura_em_celsius(self) -> float:
        if not self.isCelsius:
            # Fórmula de conversão: (F - 32) * 5/9
            return (self.temperature - 32) * 5 / 9
        return self.temperature


# --- DADOS FORNECIDOS NA QUESTÃO ---
data_samples = [
    {
        "date": "2023-05-20",
        "temperature": 62.2,
        "isCelsius": False,
        "airQualityIndex": "24",
        "sunriseTime": "01:26",
        "sunsetTime": "00:00",
    },
    {
        "date": "2023-05-21",
        "temperature": "64.4",
        "isCelsius": "not true",
        "airQualityIndex": 23,
        "sunriseTime": "01:10",
        "sunsetTime": "00:16",
    },
    {
        "date": "2023-05-22",
        "temperature": 14.4,
        "airQualityIndex": 21,
    }
]


# --- PROCESSAMENTO DOS DADOS ---

temperaturas_celsius = []

for sample in data_samples:
    # O Pydantic valida os tipos primitivos automaticamente 
    obj_validado = WeatherSample(**sample)
    temperaturas_celsius.append(obj_validado.temperatura_em_celsius)

# Calculando a média aritmética simples
temperatura_media = sum(temperaturas_celsius) / len(temperaturas_celsius)

# Exibindo o resultado final formatado com duas casas decimais
print(f"Temperaturas em Celsius convertidas/validadas: {temperaturas_celsius}")
print(f"Temperatura média em Murmansk: {temperatura_media:.2f}°C")

#### Q3
> Adaptada desse [tutorial de Pydantic](https://github.com/adonath/scipy-2023-pydantic-tutorial/tree/main) criado por [Axel Donath](https://github.com/adonath) e [Nick Langellier](https://github.com/nlangellier).

Na célula a seguir, coletamos dados reais de uma das principais APIs de previsão do tempo, [open-meteo](https://open-meteo.com/en/docs). Não se preocupe em entender esse código, o mais importante é entender o resultado que ele retorna, ilustrado a seguir para uma coleta da temperatura dos últimos 15 dias em Itabira -MG. Caso deseje alterar a cidade de coleta, basta alimentar a latitude e longitude desejada, como nas opções a seguir.
* Itabira: `'latitude': -19.656655787605846, 'longitude': -43.228922960534476`
* Bom Despacho: `'latitude': -19.726308457732443, 'longitude': -45.27462803349767`

```python
{
  "latitude": -19.5,
  "longitude": -43.375,
  "generationtime_ms": 0.01800060272216797,
  "utc_offset_seconds": 0,
  "timezone": "GMT",
  "timezone_abbreviation": "GMT",
  "elevation": 2.0,
  "hourly_units": {
    "time": "iso8601",
    "temperature_2m": "\u00b0C"
  },
  "hourly": {
    "time": [
      "2024-07-19T00:00",
      "2024-07-19T01:00",
      "2024-07-19T02:00",
      ...
    ],
    "temperature_2m": [
      21.9,
      20.9,
      20.0,
      ... 
    ]
  }
}
```

Você deve escrever um modelo Pydantic `OpenMeteo` que receba diretamente a resposta dessa API, através do comando:
```python
dados = OpenMeteo(**response)
``` 

Para comportar a estrutura hierárquica desse dicionário (é um dicionário com alguns dicionários internos), você deve criar uma classe Pydantic para cada dicionário interno (`HourlyUnits` e `Hourly`), com seus respectivos atributos. Essas classes serão atributos da classe principal `OpenMeteo`, que terá também os outros atributos da resposta (`latitude`, `longitude`, etc.).



In [ ]:
import requests, json

url = 'https://api.open-meteo.com/v1/forecast'
lat, long = -19.656655787605846, -43.228922960534476
params = {'latitude': lat, 'longitude': long, 'elevation': 2,
          'hourly': 'temperature_2m', 'forecast_days': 15}
response = requests.get(url, params=params).json()
print(json.dumps(response, indent=2))

In [ ]:
#### Escreva aqui seus modelos Pydantic
from typing import List
from pydantic import BaseModel, Field

# 1. Modelo para o dicionário interno 'hourly_units'
class HourlyUnits(BaseModel):
    time: str
    temperature_2m: str

# 2. Modelo para o dicionário interno 'hourly'
class Hourly(BaseModel):
    time: List[str]
    temperature_2m: List[float]

# 3. Modelo principal que unifica toda a resposta da API
class OpenMeteo(BaseModel):
    latitude: float
    longitude: float
    generationtime_ms: float
    utc_offset_seconds: int
    timezone: str
    timezone_by_id: str = Field(default="GMT", alias="timezone_abbreviation") # Trata a abreviação do JSON
    elevation: float
    
    # Aninhando os submodelos criados anteriormente
    hourly_units: HourlyUnits
    hourly: Hourly

    class Config:
        # Permite mapear os aliases (como timezone_abbreviation) corretamente ao desempacotar com **
        populate_by_name = True

#### Q4. 

Com os dados carregados na questão anterior plote um gráfico de linha, com a biblioteca de sua preferência, onde o eixo `x` são os timestamps (data e hora) e o eixo `y` é a temperatura medida.

In [ ]:
#### Escreva aqui a sua resposta
import matplotlib.pyplot as plt

# Configurando o tamanho da figura
plt.figure(figsize=(14, 6))

# Plotando o gráfico de linha com os dados do objeto Pydantic validado anteriormente
plt.plot(dados.hourly.time, dados.hourly.temperature_2m, color='tab:blue', linewidth=2, label='Temperatura (°C)')

# Configurando títulos e rótulos dos eixos
plt.title('Evolução da Temperatura dos Últimos 15 Dias (OpenMeteo)', fontsize=14, fontweight='bold')
plt.xlabel('Timestamp (Data e Hora)', fontsize=12)
plt.ylabel('Temperatura Medida (°m)', fontsize=12)

# Ajustando a rotação dos timestamps no eixo X para não sobrepor o texto
plt.xticks(rotation=45, ha='right')

# Como são muitos pontos (15 dias de hora em hora), vamos mostrar apenas alguns rótulos no eixo X para ficar legível
ax = plt.gca()
ax.set_xticks(ax.get_xticks()[::24])  # Mostra um rótulo a cada 24 horas (1 dia)

# Ativando a grade de fundo e a legenda
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

# Exibindo o gráfico de forma limpa
plt.tight_layout()
plt.show()